In [1]:
print("Hello World!")

Hello World!


In [3]:
import warnings
import pandas as pd

# Suppress Pandas SettingWithCopyWarning
pd.options.mode.chained_assignment = None

# Suppress standard Python warnings
warnings.filterwarnings('ignore')

import requests
url = "https://fantasy.premierleague.com/api/bootstrap-static/"
res = requests.get(url).json()
df = pd.DataFrame(res['teams'])
df.columns

team_cols = [
    'id',
    'short_name',
    'position',
    'played'
]
team_df = df[team_cols]
team_df.head()
# Fix slice warning by making an explicit copy
team_df = df[team_cols].copy()

# Fetch fixture data
fixtures_url = "https://fantasy.premierleague.com/api/fixtures/"
fixtures_data = requests.get(fixtures_url).json()
fixtures_df = pd.DataFrame(fixtures_data)

# Filter for unplayed upcoming fixtures
upcoming_fixtures = fixtures_df[fixtures_df['finished'] == False].copy()

# Create lookup mapping for team names/short names
team_id_to_short = dict(zip(df['id'], df['short_name']))

# Build dictionary to store next 5 fixtures per team
team_next_5 = {team_id: [] for team_id in df['id']}

# Iterate through upcoming matches and store home/away fixture details
for _, fix in upcoming_fixtures.iterrows():
    h_id, a_id = fix['team_h'], fix['team_a']
    h_fdr, a_fdr = fix['team_h_difficulty'], fix['team_a_difficulty']

    if len(team_next_5[h_id]) < 5:
        team_next_5[h_id].append((f"{team_id_to_short[a_id]}(H) ({h_fdr})", h_fdr))

    if len(team_next_5[a_id]) < 5:
        team_next_5[a_id].append((f"{team_id_to_short[h_id]}(A) ({a_fdr})", a_fdr))

# Extract F1-F5 strings and calculate cumulative difficulty score safely using .loc
for i in range(5):
    team_df.loc[:, f'F{i+1}'] = team_df['id'].map(lambda tid: team_next_5[tid][i][0] if len(team_next_5[tid]) > i else None)

team_df.loc[:, 'fdr_sum_next_5'] = team_df['id'].map(lambda tid: sum(fix[1] for fix in team_next_5[tid][:5]))

team_id_to_short = dict(zip(team_df['id'], team_df['short_name']))
team_df.head(10)

,id,short_name,position,played,F1,F2,F3,F4,F5,fdr_sum_next_5
0,1,ARS,2,0,LEE(H) (2),NFO(A) (3),EVE(H) (3),LIV(A) (4),HUL(H) (2),14
1,2,AVL,16,0,BRE(H) (3),NEW(A) (3),MCI(H) (4),FUL(H) (2),MUN(A) (4),16
2,3,BOU,17,0,CHE(A) (4),SUN(H) (2),MUN(A) (4),LEE(H) (2),IPS(A) (2),14
3,4,BRE,4,0,AVL(A) (4),LIV(H) (4),HUL(A) (2),NFO(H) (3),BHA(A) (3),16
4,5,BHA,3,0,SUN(A) (3),CRY(H) (3),LIV(A) (4),MCI(A) (5),BRE(H) (3),18
5,6,CHE,10,0,BOU(H) (3),EVE(A) (3),TOT(H) (3),MUN(H) (4),SUN(A) (3),16
6,7,COV,18,0,NEW(H) (2),TOT(A) (3),FUL(H) (2),SUN(H) (2),EVE(A) (3),12
7,8,CRY,15,0,NFO(H) (3),BHA(A) (3),NEW(H) (2),TOT(A) (3),LIV(H) (4),15
8,9,EVE,7,0,HUL(A) (2),CHE(H) (4),ARS(A) (5),NEW(A) (3),COV(H) (2),16
9,10,FUL,19,0,IPS(A) (2),HUL(H) (2),COV(A) (2),AVL(A) (4),NEW(H) (2),12
